In [3]:
import os
from dotenv import load_dotenv

In [4]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [5]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [6]:
%pwd

'c:\\Users\\91955\\OneDrive\\Desktop\\question_generator'

# DOCUMENT LOADING #


In [7]:
from langchain_community.document_loaders import PyPDFLoader

In [8]:
file_path = "C:/Users/91955/OneDrive/Desktop/question_generator/rag.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()
print(data[0].page_content)

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All
rights reserved. Draft of January 6, 2026.
CHAPTER
11
Information Retrieval and
Retrieval-Augmented Generation
On two occasions I have been asked,—“Pray, Mr. Babbage, if you put into
the machine wrong ﬁgures, will the right answers come out?” ... I am not able
rightly to apprehend the kind of confusion of ideas that could provoke such a
question. Babbage (1864)
People need to know things. So pretty much as soon as there were computers
we were asking them questions. By 1961 there was a system to answer questions
about American baseball statistics like “How many games did the Yankees play
in July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep
Thought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,
answered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And
because so much knowledge is encoded in text, systems were answering questions
at h

In [9]:
len(data)

25

In [10]:
question_gen=""
for page in data:
    question_gen+=page.page_content


In [11]:
question_gen

'Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All\nrights reserved. Draft of January 6, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong ﬁgures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty much as soon as there were computers\nwe were asking them questions. By 1961 there was a system to answer questions\nabout American baseball statistics like “How many games did the Yankees play\nin July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep\nThought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,\nanswered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And\nbecause so much knowledge is encoded in text, systems were answeri

# CHUNKING #

In [12]:
from langchain_text_splitters import TokenTextSplitter

In [13]:
splitter_ques_gen=TokenTextSplitter(
    model_name="gpt-3.5-turbo",
    chunk_size=10000,
    chunk_overlap=200)

In [14]:
chunks_ques_gen=splitter_ques_gen.split_text(question_gen)

In [15]:
chunks_ques_gen

["Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All\nrights reserved. Draft of January 6, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong ﬁgures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty much as soon as there were computers\nwe were asking them questions. By 1961 there was a system to answer questions\nabout American baseball statistics like “How many games did the Yankees play\nin July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep\nThought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,\nanswered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And\nbecause so much knowledge is encoded in text, systems were answer

In [16]:
len(chunks_ques_gen)

2

In [17]:
type(chunks_ques_gen)

list

In [18]:
from langchain_core.documents import Document


In [19]:
documents_ques_gen=[Document(page_content=chunk) for chunk in chunks_ques_gen]

In [20]:
documents_ques_gen

[Document(page_content="Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All\nrights reserved. Draft of January 6, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong ﬁgures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty much as soon as there were computers\nwe were asking them questions. By 1961 there was a system to answer questions\nabout American baseball statistics like “How many games did the Yankees play\nin July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep\nThought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,\nanswered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And\nbecause so much knowledge is encoded in tex

In [21]:
type(documents_ques_gen[0])

langchain_core.documents.base.Document

In [49]:
splitter = TokenTextSplitter(model_name="gpt-3.5-turbo", chunk_size=500, chunk_overlap=50)

In [50]:
document_ans_gen=splitter.split_documents(documents_ques_gen)

In [51]:
document_ans_gen

[Document(page_content='Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All\nrights reserved. Draft of January 6, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong ﬁgures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty much as soon as there were computers\nwe were asking them questions. By 1961 there was a system to answer questions\nabout American baseball statistics like “How many games did the Yankees play\nin July?” (Green et al., 1961). Even ﬁctional computers in the 1970s like Deep\nThought, invented by Douglas Adams in The Hitchhiker’s Guide to the Galaxy ,\nanswered “the Ultimate Question Of Life, The Universe, and Everything”. 1 And\nbecause so much knowledge is encoded in tex

In [52]:
len(document_ans_gen)

43

# GROQ API KEY #

In [53]:
from langchain_groq import ChatGroq

In [54]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# PROMPT TEMPLATE #

In [42]:
prompt_template = """
You are an expert AI interviewer and question generation assistant.

Your task is to generate high-quality interview and exam preparation questions 
ONLY from the provided context.

Context:
----------------
{text}
----------------

Instructions:
- Generate clear and professional questions.
- Cover important concepts from the context.
- Include beginner, intermediate, and advanced questions.
- Include coding questions if the topic supports programming.
- Avoid duplicate questions.
- Do not generate questions outside the provided context.
- Preserve important technical information.

Output Format:
1.
2.
3.

QUESTIONS:
"""

In [43]:
from langchain_core.prompts import PromptTemplate

In [44]:
PROMPT = PromptTemplate.from_template(prompt_template)

In [45]:
from langchain.chains.summarize import load_summarize_chain

In [46]:
question_generation_chain = load_summarize_chain(
    llm,
    chain_type="map_reduce",
    map_prompt=PROMPT,
    combine_prompt=PROMPT
)

In [55]:
ques=question_generation_chain.run(documents_ques_gen)
print(ques)

1. What is the primary issue with relying solely on large language models for knowledge question answering, and how does retrieval-augmented generation (RAG) address this limitation?

2. Describe the vector space model in information retrieval, including how documents and queries are represented as vectors, and explain the role of term weights such as tf-idf in this model.

3. Compare and contrast sparse and dense vectors in information retrieval, providing examples of each and discussing how they impact the retrieval process.

4. Write a Python function to calculate the tf-idf score for a given term in a document, assuming access to a list of documents and a list of terms.

5. Explain the concept of an inverted index in information retrieval, including its two main components and how it facilitates efficient document retrieval based on specific terms.

6. What is the purpose of stop words in information retrieval, and provide an example of how a stop word affects a query, including it